# Training a CNN on MNIST with PyTorch Lightning

This notebook demonstrates how to train a simple, efficient Convolutional Neural Network (CNN) with residual connections on the MNIST dataset using PyTorch Lightning. The configuration is tuned for ≥99.7% accuracy.

In [1]:
!pip install pytorch-lightning torch torchvision datasets wandb pillow scikit-learn tsilva_notebook_utils==0.0.66

In [2]:
from PIL import ExifTags, Image
Image.ExifTags = ExifTags  # Hack to bypass broken import

Define config:

In [3]:
import os

def setup_config():
    # General Settings
    model_id = "resnet18"#"resnet50"
    backbone_warmup_percentage = 0.1
    dataset_id = "cifar10"  # Options: "mnist" or "cifar10"
    pretrained_dataset_id = "imagenet"
    seed = 42
    n_epochs = 100
    batch_size = 256
    # Optimizer Settings
    learning_rate = 1e-3
    weight_decay = 0
    # Model Architecture
    nonlinearity = 'relu'
    # Data Settings
    train_size = 0.8
    val_size = 0.2
    # Softcoded new params
    label_smoothing = 0.0
    early_stopping_patience = 7
    early_stopping_min_delta = 1e-4
    precision = "16-mixed"
    swa_lrs = 0#1e-2
    activation = "relu"
    
    augmentation_pipeline = [
        ("RandomCrop", [], dict(size=32, padding=4)),
        ("RandomHorizontalFlip", [], dict(p=0.5)),
        ("ColorJitter", [], dict(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)),
        ("RandomErasing", [], dict(p=0.25, scale=(0.02, 0.33), ratio=(0.3, 3.3))),
    ]

    # TODO: softcode this
    os.environ["NOTEBOOK_ID"] = "mnist-cnn-pl"

    return {
        'model_id': model_id,
        "pretrained_dataset_id": pretrained_dataset_id,
        'backbone_warmup_percentage' : backbone_warmup_percentage,
        'dataset_id': dataset_id,
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'nonlinearity': nonlinearity,
        'weight_decay': weight_decay,
        'train_size': train_size,
        'val_size': val_size,
        'label_smoothing': label_smoothing,
        'early_stopping_patience': early_stopping_patience,
        'early_stopping_min_delta': early_stopping_min_delta,
        'precision': precision,
        'swa_lrs': swa_lrs,
        'activation': activation,
        'augmentation_pipeline': augmentation_pipeline
    }

CONFIG = setup_config()

Set seed for reproducibility:

In [4]:
import pytorch_lightning as pl
pl.seed_everything(CONFIG['seed'], workers=True)

Seed set to 42


42

Set matmul precision:

In [5]:
import torch
torch.set_float32_matmul_precision('high')

Login to wandb:

In [6]:
import wandb
wandb.login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from torchvision import transforms

# Dataset specifications with only essential info
DATASET_SPECS = {
    "imagenet": {
        "image_size": 224,
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "cifar10": {
        "image_size": 32,
        "mean": [0.4914, 0.4822, 0.4465],
        "std": [0.2023, 0.1994, 0.2010]
    },
    "mnist": {
        "image_size": 28,
        "mean": [0.1307],
        "std": [0.3081]
    }
}

def build_dataset_transforms(dataset_id: str, augmentation_pipeline: list = []):
    assert dataset_id in DATASET_SPECS, f"Unknown dataset spec: {dataset_id}"
    spec = DATASET_SPECS[dataset_id]

    image_size = spec["image_size"]
    crop_fraction = 0.875
    resize_size = int(round(image_size / crop_fraction))

    preprocessing_pipeline = [
        transforms.Resize(resize_size),
        transforms.CenterCrop(image_size)
    ]
    
    _augmentation_pipeline = []
    for name, args, kwargs in augmentation_pipeline:
        _class = getattr(transforms, name, None)
        augmentation_fn = _class(*args, **kwargs)
        _augmentation_pipeline.append(augmentation_fn)

    normalization_pipeline = [
        transforms.ToTensor(),
        transforms.Normalize(mean=spec["mean"], std=spec["std"]),
    ]

    train_transform = transforms.Compose(
        preprocessing_pipeline + _augmentation_pipeline + normalization_pipeline
    )
    test_transform = transforms.Compose(
        preprocessing_pipeline + normalization_pipeline
    )

    return train_transform, test_transform


Define MNIST data module:

In [8]:
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader

class MNISTDataModule(pl.LightningDataModule):
    def __init__(
        self, 
        batch_size, 
        train_size, 
        seed, 
        num_workers=2, 
        train_shuffle=True,
        val_shuffle=False,
        test_shuffle=False,
        augmentation_pipeline=[],
        pretrained_dataset_id=None
    ):
        super().__init__()
        self.dataset_id = "mnist"
        self.download_path = f"./temp/{self.dataset_id}"
        self.batch_size = batch_size
        self.train_size = train_size
        self.seed = seed
        self.num_workers = num_workers
        self.train_shuffle = train_shuffle
        self.val_shuffle = val_shuffle
        self.test_shuffle = test_shuffle
        self.n_classes = 10
        self.augmentation_pipeline = augmentation_pipeline
        self.pretrained_dataset_id = pretrained_dataset_id


    def prepare_data(self):
        MNIST(root=self.download_path, train=True, download=True)
        MNIST(root=self.download_path, train=False, download=True)

    def setup(self, stage=None):
        dataset_id = self.pretrained_dataset_id if self.pretrained_dataset_id else self.dataset_id
        self.train_transform, self.test_transform = build_dataset_transforms(dataset_id, self.augmentation_pipeline)

        full = MNIST(root=self.download_path, train=True, transform=self.train_transform)
        total = len(full)
        train_size = int(total * self.train_size)
        val_size = total - train_size
        self.train_set, self.val_set = torch.utils.data.random_split(
            full, [train_size, val_size], generator=torch.Generator().manual_seed(self.seed)
        )
        self.val_set.dataset.transform = self.test_transform
        self.test_set = MNIST(root=self.download_path, train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(
            self.train_set, 
            batch_size=self.batch_size, 
            shuffle=self.train_shuffle, 
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_set, 
            batch_size=self.batch_size, 
            shuffle=self.val_shuffle, 
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_set, 
            batch_size=self.batch_size, 
            shuffle=self.test_shuffle, 
            num_workers=self.num_workers
        )

Define CIFAR10 data module:

In [9]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

class CIFAR10DataModule(pl.LightningDataModule):
    def __init__(
        self, 
        batch_size, 
        train_size, 
        seed, 
        num_workers=2, 
        train_shuffle=True,
        val_shuffle=False,
        test_shuffle=False,
        augmentation_pipeline=[],
        pretrained_dataset_id=None
    ):
        super().__init__()
        self.dataset_id = "cifar10"
        self.download_path = f"./temp/{self.dataset_id}"
        self.batch_size = batch_size
        self.train_size = train_size
        self.seed = seed
        self.num_workers = num_workers
        self.train_shuffle = train_shuffle
        self.val_shuffle = val_shuffle
        self.test_shuffle = test_shuffle
        self.n_classes = 10
        self.augmentation_pipeline = augmentation_pipeline
        self.pretrained_dataset_id = pretrained_dataset_id

    def prepare_data(self):
        CIFAR10(root=self.download_path, train=True, download=True)
        CIFAR10(root=self.download_path, train=False, download=True)

    def setup(self, stage=None):
        dataset_id = self.pretrained_dataset_id if self.pretrained_dataset_id else self.dataset_id
        self.train_transform, self.test_transform = build_dataset_transforms(dataset_id, self.augmentation_pipeline)

        full = CIFAR10(root=self.download_path, train=True, transform=self.train_transform)
        total = len(full)
        train_size = int(total * self.train_size)
        val_size = total - train_size
        self.train_set, self.val_set = torch.utils.data.random_split(
            full, [train_size, val_size], generator=torch.Generator().manual_seed(self.seed)
        )
        self.val_set.dataset.transform = self.test_transform
        self.test_set = CIFAR10(root=self.download_path, train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(
            self.train_set, 
            batch_size=self.batch_size, 
            shuffle=self.train_shuffle, 
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_set, 
            batch_size=self.batch_size, 
            shuffle=self.val_shuffle, 
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_set, 
            batch_size=self.batch_size, 
            shuffle=self.test_shuffle, 
            num_workers=self.num_workers
        )



Create the data module:

In [16]:
def create_data_module(config, **kwargs):
    dataset_id = config['dataset_id']
    dataset_modules = {
        "mnist": MNISTDataModule,
        "cifar10": CIFAR10DataModule
    }
    dataset_id = dataset_id.lower()
    datamodule_class = dataset_modules.get(dataset_id)
    assert datamodule_class is not None, f"Unsupported dataset: {dataset_id}"
    datamodule = datamodule_class(**{
        "seed": config['seed'],
        "batch_size": config['batch_size'],
        "train_size": config['train_size'],
        "augmentation_pipeline": config['augmentation_pipeline'],
        "pretrained_dataset_id": config['pretrained_dataset_id'],
        **kwargs
    })
    datamodule.prepare_data()
    return datamodule

dm = create_data_module(CONFIG)
dm

Files already downloaded and verified
Files already downloaded and verified


Create the model:

In [17]:
from torch import nn

class LitModel(pl.LightningModule):
    def __init__(self, lr=None, n_classes=None):
        super().__init__()
        self.save_hyperparameters()

        assert n_classes is not None, "n_classes must be provided"
        
        self.model = torch.hub.load("pytorch/vision", CONFIG["model_id"], weights="DEFAULT")
        
        self.model.fc = nn.Linear(self.model.fc.in_features, n_classes)

        self.criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
    
    def freeze_backbone(self):
        for param in self.model.parameters(): param.requires_grad = False
        for param in self.model.fc.parameters(): param.requires_grad = True

    def unfreeze_backbone(self):
        for param in self.model.parameters(): param.requires_grad = True
        for param in self.model.fc.parameters(): param.requires_grad = True

    def forward(self, x):
        x = self.model(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train/loss", loss, prog_bar=True)
        self.log("train/acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("val/loss", loss, prog_bar=True, sync_dist=True)
        self.log("val/acc", acc, prog_bar=True, sync_dist=True)
        return {}

    def on_validation_epoch_end(self):
        pass

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test/loss", loss)
        self.log("test/acc", acc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            # TODO: why not only self.hparams.lr
            lr=self.hparams.lr if hasattr(self.hparams, 'lr') and self.hparams.lr is not None else CONFIG['learning_rate'],
            weight_decay=CONFIG['weight_decay']
        )
        # TODO: softcode reduce lr on plateau params
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val/loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
model

Using cache found in /home/tsilva/.cache/torch/hub/pytorch_vision_main


LitModel(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runn

Find optimal batch size:

In [ ]:
#from pytorch_lightning.tuner import Tuner
#trainer = pl.Trainer()
#tuner = Tuner(trainer)
#tuner.scale_batch_size(model, datamodule=dm, mode="power")

First try to overfit a batch to make sure the training pipeline works:

In [18]:
from tsilva_notebook_utils.lightning import ThresholdStoppingCallback

trainer = pl.Trainer(
    log_every_n_steps=1,
    overfit_batches=1,
    max_epochs=100,
    callbacks=[ThresholdStoppingCallback("train/acc", 1.0)]
)
model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
trainer.fit(model, datamodule=create_data_module(CONFIG, batch_size=32, train_shuffle=False))

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.


Using cache found in /home/tsilva/.cache/torch/hub/pytorch_vision_main


Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | ResNet           | 11.2 M | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Stopping training as train/acc reached 1.0


Run the training loop:

In [ ]:
from tsilva_notebook_utils.lightning import EpochTimeLogger, BackboneWarmupCallback
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import Timer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks import StochasticWeightAveraging

trainer_callbacks = [
    Timer(), 
    EpochTimeLogger(), 
    ModelCheckpoint(
        monitor="val/loss",
        save_top_k=1,
        mode="min",
        save_last=True,
        dirpath="temp/checkpoints",
        filename=os.environ["NOTEBOOK_ID"] + "-{epoch:02d}-{val/loss:.2f}"
    ), 
    LearningRateMonitor(logging_interval="epoch"),
    EarlyStopping(
        monitor="val/loss",
        patience=CONFIG['early_stopping_patience'],
        min_delta=CONFIG['early_stopping_min_delta'],
        mode="min",
        verbose=True,
        strict=True
    ) if CONFIG['early_stopping_patience'] > 0 else None,
    StochasticWeightAveraging(swa_lrs=CONFIG['swa_lrs']) if CONFIG['swa_lrs'] > 0 else None,
    BackboneWarmupCallback(CONFIG['backbone_warmup_percentage']) if CONFIG['backbone_warmup_percentage'] > 0 else None
]
trainer_callbacks = [cb for cb in trainer_callbacks if cb is not None]

num_gpus = torch.cuda.device_count()
trainer = pl.Trainer(
    devices=num_gpus,
    accelerator="auto",
    strategy="auto",
    benchmark=True,
    max_epochs=CONFIG['n_epochs'],
    log_every_n_steps=5,
    logger=WandbLogger(project=os.environ["NOTEBOOK_ID"], config=CONFIG),
    callbacks=trainer_callbacks,
    precision=CONFIG['precision']
)
model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
trainer.fit(model, datamodule=dm)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Using cache found in /home/tsilva/.cache/torch/hub/pytorch_vision_main


Files already downloaded and verified
Files already downloaded and verified


/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/tsilva/repos/tsilva/aiml-notebooks/notebooks/temp/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | ResNet           | 11.2 M | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

[Epoch 0] Training with frozen backbone until epoch 10...


Calculate the post-training test set accuracy:

In [ ]:
single_gpu_trainer = pl.Trainer(
    devices=1,
    accelerator="auto"
)
single_gpu_trainer.test(model, datamodule=dm)